# Phase 3: Final Model Evaluation
**Author:** S.L Sarma (Group Leader)

In this notebook, we mathematically evaluate the 6 clustering models trained by Manathunge and Ahamed to objectively determine which model is best for our ecommerce business.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import os

os.makedirs('outputs/figures', exist_ok=True)

print("Loading data...")
# Load the original scaled data
try:
    df_rfm = pd.read_csv('outputs/2_rfm_data_herath.csv')
    X = df_rfm[['Scaled_R', 'Scaled_F', 'Scaled_M']]
except FileNotFoundError:
    print("Error: Could not find outputs/2_rfm_data_herath.csv")
    exit()

# Load all the prediction CSVs
print("Loading model predictions...")
try:
    df_kmeans = pd.read_csv('outputs/predictions/model1_kmeans_predictions.csv')
    df_mini = pd.read_csv('outputs/predictions/model2_minibatch_predictions.csv')
    df_gmm = pd.read_csv('outputs/predictions/model3_gmm_predictions.csv')
    df_agg = pd.read_csv('outputs/predictions/model4_agglomerative_predictions.csv')
    df_dbscan = pd.read_csv('outputs/predictions/model5_dbscan_predictions.csv')
    df_birch = pd.read_csv('outputs/predictions/model6_birch_predictions.csv')
except Exception as e:
    print(f"Error loading prediction CSVs: {e}")
    print("Please make sure all team members have generated their CSV files in outputs/predictions/")
    exit()

# Define the models and their labels
models = {
    'K-Means': df_kmeans.iloc[:, 1].values,
    'MiniBatch K-Means': df_mini.iloc[:, 1].values,
    'Gaussian Mixture': df_gmm.iloc[:, 1].values,
    'Agglomerative': df_agg.iloc[:, 1].values,
    'DBSCAN': df_dbscan.iloc[:, 1].values,
    'Birch': df_birch.iloc[:, 1].values
}

results = []

print("\nCalculating Metrics for all 6 Models (This may take a minute)...")
for name, labels in models.items():
    # Only calculate if the model didn't just assign everything to 1 cluster (e.g. DBSCAN noise failure)
    if len(np.unique(labels)) > 1:
        sil = silhouette_score(X, labels)
        db = davies_bouldin_score(X, labels)
        ch = calinski_harabasz_score(X, labels)
        results.append({'Model': name, 'Silhouette': sil, 'Davies_Bouldin': db, 'Calinski_Harabasz': ch})
        print(f"[{name}] - Silhouette: {sil:.4f} | Davies-Bouldin: {db:.4f}")
    else:
        print(f"[{name}] - Failed to find multiple clusters.")
        results.append({'Model': name, 'Silhouette': 0, 'Davies_Bouldin': 0, 'Calinski_Harabasz': 0})

df_results = pd.DataFrame(results)
df_results.to_csv('outputs/predictions/model_evaluation_metrics.csv', index=False)

# Plotting the winning Silhouette Scores
plt.figure(figsize=(12, 6))
sns.set_theme(style="whitegrid")
bars = sns.barplot(x='Silhouette', y='Model', data=df_results.sort_values('Silhouette', ascending=False), palette='viridis')

plt.title('Silhouette Score Comparison (Higher is Better)', fontsize=16, fontweight='bold')
plt.xlabel('Silhouette Score', fontsize=12)
plt.ylabel('Clustering Model', fontsize=12)

# Highlight K-Means if it wins
for i, bar in enumerate(bars.patches):
    if df_results.sort_values('Silhouette', ascending=False).iloc[i]['Model'] == 'K-Means':
        bar.set_color('crimson')

plt.tight_layout()
plt.savefig('outputs/figures/model_comparison_silhouette.png', dpi=300)
print("\nEvaluation complete! Saved metrics to CSV and generated the comparison chart.")
